# TCN_MLP_ENSEMBLE (Lightweight - 9 Climate Features) - Trimmed Training Notebook

This notebook contains only the minimal code to run the 5-fold train/test training, ensemble voting, and the final summary.

In [1]:
# Imports and reproducibility
import os
import random
import json
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers, Model, Input, optimizers
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error
import matplotlib.pyplot as plt

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print('Imports ready. Seed set to', SEED)

Imports ready. Seed set to 42


In [2]:
# Load processed dataset and build climate sequences (9 features)
DATA_PATH = '../data/processed_dataset.csv'
df = pd.read_csv(DATA_PATH)
df = df.dropna(subset=['Yield_kg_per_ha']).copy()

climate_features = [
    'T2M', 'T2M_MAX', 'T2M_MIN', 'TS', 'T2MDEW', 'T2MWET', 'PRECTOTCORR', 'RH2M', 'QV2M'
]
"""
Category,       Parameter_Code, Parameter_Name,         Unit
temperature,    T2M,            Mean temperature at 2m, °C
temperature,    T2M_MAX,        Max temperature at 2m,  °C
temperature,    T2M_MIN,        Min temperature at 2m,  °C
temperature,    TS,             land surface temperature,°C
rainfall,       PRECTOTCORR,    Bias-corrected total precipitation,mm/day
humidity,       RH2M,           Relative humidity at 2m,%
humidity,       QV2M,           Specific humidity at 2m,g/kg
humidity,       T2MDEW,         Dew point temperature at 2m,°C
humidity,       T2MWET,         Wet bulb temperature at 2m,°C
"""
n_features = len(climate_features)
n_months = 12

# Build X_seq tensor
X_seq = np.zeros((df.shape[0], n_months, n_features), dtype=np.float32)
for m in range(1, n_months + 1):
    for i, feat in enumerate(climate_features):
        X_seq[:, m - 1, i] = df[f'{feat}_m{m}'].values

y_raw = df['Yield_kg_per_ha'].values.astype(np.float32)
years = df['Year'].values.astype(np.float32)
region_to_id = {r: i for i, r in enumerate(sorted(df['Region'].unique()))}
crop_to_id = {c: i for i, c in enumerate(sorted(df['Crop'].unique()))}
region_ids = np.array([region_to_id[r] for r in df['Region'].values], dtype=np.int32)
crop_ids = np.array([crop_to_id[c] for c in df['Crop'].values], dtype=np.int32)

print('Built X_seq with shape:', X_seq.shape)
print('Dataset samples:', df.shape[0])

Built X_seq with shape: (600, 12, 9)
Dataset samples: 600


In [3]:
# Stratified labels and year/context feature builder
epsilon = 1e-6
y_log = np.log(y_raw + epsilon)
strat_labels = np.array([f'{c}_{r}' for c, r in zip(crop_ids, region_ids)])
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

def build_year_features(year_values):
    # Return only time-related features (normalized year, sin, cos).
    # Region and crop identity should be passed as separate inputs (embeddings).
    y_norm = ((year_values.reshape(-1, 1) - 1999.0) / 24.0).astype(np.float32)
    y_sin = np.sin(2.0 * np.pi * y_norm).astype(np.float32)
    y_cos = np.cos(2.0 * np.pi * y_norm).astype(np.float32)
    return np.column_stack([y_norm, y_sin, y_cos]).astype(np.float32)

print('Prepared stratified labels and year feature builder')

Prepared stratified labels and year feature builder


In [4]:
# Model definition (TCN-MLP)
def build_model(n_regions, n_crops, n_year_features, n_features, l2_reg=1e-3, dropout=0.12, lr=3e-4, weight_decay=3e-4):
    sequence_input = Input(shape=(12, n_features), name='sequence_input')
    region_input = Input(shape=(1,), dtype='int32', name='region_input')
    crop_input = Input(shape=(1,), dtype='int32', name='crop_input')
    year_input = Input(shape=(n_year_features,), name='year_input')
    x = layers.Conv1D(48, kernel_size=3, padding='causal', activation='relu', kernel_regularizer=regularizers.l2(l2_reg))(sequence_input)
    x = layers.BatchNormalization(momentum=0.95, epsilon=1e-3)(x)
    x = layers.Dropout(dropout)(x)
    x_dil1 = layers.Conv1D(48, kernel_size=3, padding='causal', activation='relu', dilation_rate=2, kernel_regularizer=regularizers.l2(l2_reg))(x)
    x_dil1 = layers.BatchNormalization(momentum=0.95, epsilon=1e-3)(x_dil1)
    x_dil1 = layers.Dropout(dropout)(x_dil1)
    x = layers.Add()([x, x_dil1])
    x_dil2 = layers.Conv1D(48, kernel_size=3, padding='causal', activation='relu', dilation_rate=4, kernel_regularizer=regularizers.l2(l2_reg))(x)
    x_dil2 = layers.BatchNormalization(momentum=0.95, epsilon=1e-3)(x_dil2)
    x_dil2 = layers.Dropout(dropout)(x_dil2)
    x = layers.Add()([x, x_dil2])
    x = layers.Conv1D(32, kernel_size=1, activation='relu', kernel_regularizer=regularizers.l2(l2_reg))(x)
    x_mean = layers.GlobalAveragePooling1D()(x)
    x_max = layers.GlobalMaxPooling1D()(x)
    x_tcn = layers.Concatenate()([x_mean, x_max])
    region_emb = layers.Embedding(input_dim=n_regions, output_dim=4)(region_input)
    region_emb = layers.Flatten()(region_emb)
    crop_emb = layers.Embedding(input_dim=n_crops, output_dim=4)(crop_input)
    crop_emb = layers.Flatten()(crop_emb)
    year_branch = layers.Dense(16, activation='relu', kernel_regularizer=regularizers.l2(l2_reg))(year_input)
    year_branch = layers.Dropout(dropout)(year_branch)
    merged = layers.Concatenate()([x_tcn, region_emb, crop_emb, year_branch])
    dense_1 = layers.Dense(40, activation='relu', kernel_regularizer=regularizers.l2(l2_reg))(merged)
    dense_1 = layers.BatchNormalization(momentum=0.95, epsilon=1e-3)(dense_1)
    dense_1 = layers.Dropout(dropout)(dense_1)
    dense_2 = layers.Dense(20, activation='relu', kernel_regularizer=regularizers.l2(l2_reg))(dense_1)
    dense_2 = layers.Dropout(dropout)(dense_2)
    output = layers.Dense(1, activation='linear')(dense_2)
    model = Model(inputs=[sequence_input, region_input, crop_input, year_input], outputs=output)
    optimizer = optimizers.AdamW(learning_rate=lr, weight_decay=weight_decay, clipnorm=1.0)
    model.compile(optimizer=optimizer, loss=tf.keras.losses.Huber(delta=0.35), metrics=['mae'])
    return model

print('Model builder ready')

Model builder ready


In [ ]:
# Run 5-fold CV with train/test splits and store models for ensemble
n_folds = 5
fold_models = {}
fold_predictions = {}
best_fold_test_r2 = -np.inf
best_fold = -1
best_model_path = '../models/TCN_MLP_ENSEMBLE.keras'

for fold, (train_idx, test_idx) in enumerate(skf.split(X_seq, strat_labels), start=1):
    print(f'\n===== Fold {fold}/{n_folds} =====')
    X_train_raw = X_seq[train_idx]
    X_test_raw = X_seq[test_idx]
    y_train_log = y_log[train_idx]
    y_test_log = y_log[test_idx]
    y_train_raw = y_raw[train_idx]
    y_test_raw = y_raw[test_idx]
    r_train = region_ids[train_idx]
    r_test = region_ids[test_idx]
    c_train = crop_ids[train_idx]
    c_test = crop_ids[test_idx]
    # Build year/context features: now only time features (norm, sin, cos)
    yr_train_raw = build_year_features(years[train_idx])
    yr_test_raw = build_year_features(years[test_idx])
    x_scaler = StandardScaler()
    X_train = x_scaler.fit_transform(X_train_raw.reshape(-1, n_features)).reshape(X_train_raw.shape).astype(np.float32)
    X_test = x_scaler.transform(X_test_raw.reshape(-1, n_features)).reshape(X_test_raw.shape).astype(np.float32)
    year_scaler = StandardScaler()
    yr_train = year_scaler.fit_transform(yr_train_raw).astype(np.float32)
    yr_test = year_scaler.transform(yr_test_raw).astype(np.float32)
    n_crops = len(crop_to_id)
    crop_log_mean = np.zeros(n_crops, dtype=np.float32)
    crop_log_std = np.zeros(n_crops, dtype=np.float32)
    for cid in range(n_crops):
        mask = c_train == cid
        vals = y_train_log[mask]
        crop_log_mean[cid] = np.mean(vals)
        crop_log_std[cid] = max(np.std(vals), 1e-3)
    y_train_norm = (y_train_log - crop_log_mean[c_train]) / crop_log_std[c_train]
    y_test_norm = (y_test_log - crop_log_mean[c_test]) / crop_log_std[c_test]
    fold_model = build_model(n_regions=len(region_to_id), n_crops=len(crop_to_id), n_year_features=yr_train.shape[1], n_features=n_features)
    fold_callbacks = [keras.callbacks.ReduceLROnPlateau(monitor='loss', factor=0.5, patience=15, min_lr=1e-5, verbose=1)]
    history = fold_model.fit([X_train, r_train, c_train, yr_train], y_train_norm, epochs=160, batch_size=32, callbacks=fold_callbacks, verbose=1)
    train_pred_norm = fold_model.predict([X_train, r_train, c_train, yr_train], verbose=0).ravel()
    train_pred_log = train_pred_norm * crop_log_std[c_train] + crop_log_mean[c_train]
    yhat_train = np.exp(train_pred_log)
    test_pred_norm = fold_model.predict([X_test, r_test, c_test, yr_test], verbose=0).ravel()
    test_pred_log = test_pred_norm * crop_log_std[c_test] + crop_log_mean[c_test]
    yhat_test = np.exp(test_pred_log)
    fold_models[fold] = fold_model
    fold_predictions[fold] = {'y_test_raw': y_test_raw, 'test_pred_norm': test_pred_norm, 'r_test': r_test, 'c_test': c_test, 'yr_test': yr_test, 'X_test': X_test, 'crop_log_mean': crop_log_mean, 'crop_log_std': crop_log_std}
    # compute test R² to track best fold
    test_r2_fold = r2_score(y_test_raw, yhat_test)
    if test_r2_fold > best_fold_test_r2:
        best_fold_test_r2 = test_r2_fold
        best_fold = fold
        fold_model.save(best_model_path)
    print(f"Completed Fold {fold}: Test R²={test_r2_fold:.4f}")

print('CV training completed. (ensemble-only reporting)')


===== Fold 1/5 =====
Epoch 1/160
15/15 ━━━━━━━━━━━━━━━━━━━━ 26s 28ms/step - loss: 0.5170 - mae: 0.9743 - learning_rate: 3.0000e-04
Epoch 2/160
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - loss: 0.4901 - mae: 0.9029 - learning_rate: 3.0000e-04
Epoch 3/160
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 0.4705 - mae: 0.8435 - learning_rate: 3.0000e-04
Epoch 4/160
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - loss: 0.4617 - mae: 0.8236 - learning_rate: 3.0000e-04
Epoch 5/160
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - loss: 0.4475 - mae: 0.7841 - learning_rate: 3.0000e-04
Epoch 6/160
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 0.4450 - mae: 0.7811 - learning_rate: 3.0000e-04
Epoch 7/160
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - loss: 0.4349 - mae: 0.7537 - learning_rate: 3.0000e-04
Epoch 8/160
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 63ms/step - loss: 0.4433 - mae: 0.7826 - learning_rate: 3.0000e-04
Epoch 9/160
15/15 ━━━━━━━━━━━━━━━━━━━━ 2s 34ms/step - loss: 0.4486 - mae: 0.8015 - learning_rate: 3.0000e

In [ ]:
# Ensemble: average predictions from all fold models on each fold's test set
ensemble_metrics = []
for fold in range(1, n_folds + 1):
    preds = fold_predictions[fold]
    y_test_raw = preds['y_test_raw']
    ensemble_pred_norms = []
    for other_fold in range(1, n_folds + 1):
        other_model = fold_models[other_fold]
        other_pred_norm = other_model.predict([preds['X_test'], preds['r_test'], preds['c_test'], preds['yr_test']], verbose=0).ravel()
        ensemble_pred_norms.append(other_pred_norm)
    ensemble_pred_norm = np.mean(ensemble_pred_norms, axis=0)
    ensemble_pred_log = ensemble_pred_norm * preds['crop_log_std'][preds['c_test']] + preds['crop_log_mean'][preds['c_test']]
    ensemble_yhat = np.exp(ensemble_pred_log)
    ensemble_r2 = r2_score(y_test_raw, ensemble_yhat)
    ensemble_mae = mean_absolute_error(y_test_raw, ensemble_yhat)
    # compute MAPE
    ensemble_mape = np.mean(np.abs((y_test_raw - ensemble_yhat) / y_test_raw)) * 100
    ensemble_metrics.append({'Fold': fold, 'Ensemble_R2': ensemble_r2, 'Ensemble_MAE': ensemble_mae, 'Ensemble_MAPE': ensemble_mape})

ensemble_df = pd.DataFrame(ensemble_metrics)
print('Ensemble completed.')

In [ ]:
# Save results and show concise final metrics (ensemble-only)
print('Training and ensemble metrics complete.')
print('\nENSEMBLE TEST RESULTS:')
print(ensemble_df.to_string(index=False))

In [ ]:
# Ensemble-only: compute MAPE, sMAPE, and MASE and per-crop ensemble metrics (no individual metrics)

def compute_mape(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

def compute_smape(y_true, y_pred):
    denom = (np.abs(y_true) + np.abs(y_pred))
    denom = np.where(denom == 0, 1.0, denom)
    return np.mean(2.0 * np.abs(y_pred - y_true) / denom) * 100

# Global crop mean baseline for MASE (naive forecast)
crop_global_means = {cid: np.mean(y_raw[crop_ids == cid]) for cid in np.unique(crop_ids)}

print('\n' + '='*80)
print('ENSEMBLE-ONLY METRICS (MAPE, sMAPE, MASE)'.center(80))
print('='*80)

ensemble_list = []
ensemble_all_y_true = []
ensemble_all_y_pred = []

for fold in range(1, n_folds + 1):
    preds = fold_predictions[fold]
    y_test_raw = preds['y_test_raw']
    c_test = preds['c_test']

    # Build ensemble predictions (average of all fold models)
    ensemble_pred_norms = []
    for other_fold in range(1, n_folds + 1):
        other_model = fold_models[other_fold]
        other_pred_norm = other_model.predict([preds['X_test'], preds['r_test'], preds['c_test'], preds['yr_test']], verbose=0).ravel()
        ensemble_pred_norms.append(other_pred_norm)
    ensemble_pred_norm = np.mean(ensemble_pred_norms, axis=0)

    # Denormalize back to yield units
    ensemble_pred_log = ensemble_pred_norm * preds['crop_log_std'][c_test] + preds['crop_log_mean'][c_test]
    ensemble_yhat = np.exp(ensemble_pred_log)

    r2 = r2_score(y_test_raw, ensemble_yhat)
    mae = mean_absolute_error(y_test_raw, ensemble_yhat)
    mape = compute_mape(y_test_raw, ensemble_yhat)
    smape = compute_smape(y_test_raw, ensemble_yhat)

    # MASE using global crop-mean naive baseline
    crop_means_arr = np.array([crop_global_means[int(c)] for c in c_test])
    naive_mae = np.mean(np.abs(y_test_raw - crop_means_arr))
    mase = mae / naive_mae if naive_mae > 0 else np.nan

    ensemble_list.append({'Fold': fold, 'Ensemble_R2': r2, 'Ensemble_MAE': mae, 'Ensemble_MAPE': mape, 'Ensemble_sMAPE': smape, 'Ensemble_MASE': mase})

    # accumulate for per-crop aggregation
    ensemble_all_y_true.append((c_test, y_test_raw))
    ensemble_all_y_pred.append((c_test, ensemble_yhat))

    print(f"Fold {fold}: R²={r2:.4f} | MAE={mae:.2f} kg/ha | MAPE={mape:.2f}% | sMAPE={smape:.2f}% | MASE={mase:.2f}")

ensemble_df = pd.DataFrame(ensemble_list)
print(f"\nEnsemble Average: R²={ensemble_df['Ensemble_R2'].mean():.4f} | MAE={ensemble_df['Ensemble_MAE'].mean():.2f} | MAPE={ensemble_df['Ensemble_MAPE'].mean():.2f}% | sMAPE={ensemble_df['Ensemble_sMAPE'].mean():.2f}% | MASE={ensemble_df['Ensemble_MASE'].mean():.2f}")

# Per-crop ensemble metrics aggregated across folds
print('\n' + '='*80)
print('PER-CROP ENSEMBLE PERFORMANCE (MAPE, sMAPE, MASE)'.center(80))
print('='*80)

id_to_crop = {v: k for k, v in crop_to_id.items()}

crop_results = []
for cid, crop_name in id_to_crop.items():
    ys_true = []
    ys_pred = []
    for (c_test_arr, y_true_arr), (c_test_arr2, y_pred_arr) in zip(ensemble_all_y_true, ensemble_all_y_pred):
        mask = c_test_arr == cid
        if np.any(mask):
            ys_true.extend(y_true_arr[mask])
            ys_pred.extend(y_pred_arr[mask])
    if ys_true:
        ys_true = np.array(ys_true)
        ys_pred = np.array(ys_pred)
        r2 = r2_score(ys_true, ys_pred)
        mae = mean_absolute_error(ys_true, ys_pred)
        mape = compute_mape(ys_true, ys_pred)
        smape = compute_smape(ys_true, ys_pred)
        naive_mae_crop = np.mean(np.abs(ys_true - crop_global_means[cid]))
        mase = mae / naive_mae_crop if naive_mae_crop > 0 else np.nan
        crop_results.append({'Crop': crop_name, 'Ens_R2': r2, 'Ens_MAE': mae, 'Ens_MAPE': mape, 'Ens_sMAPE': smape, 'Ens_MASE': mase})

crop_perf_df = pd.DataFrame(crop_results)
print(crop_perf_df.to_string(index=False))

# Save ensemble metrics including new metrics
ensemble_df.to_csv('../results/ensemble_metrics_mapes_maase_trimmed.csv', index=False)
crop_perf_df.to_csv('../results/per_crop_ensemble_mapes_maase_trimmed.csv', index=False)

print('\n' + '='*80)
print('✓ Ensemble metrics (MAPE, sMAPE, MASE) saved to CSV files')
print('='*80)


In [ ]:
# Visualize ensemble results (high-resolution and screenshot-friendly)


# Readability settings for screenshots
DISPLAY_DPI = 220   # quality inside notebook
SAVE_DPI = 400      # quality when saving image files


plt.rcParams['figure.dpi'] = DISPLAY_DPI
plt.rcParams['savefig.dpi'] = SAVE_DPI
plt.rcParams['font.size'] = 11


def row_normalize(frame):
    """Scale each row to 0-1 so colors are comparable within a row."""
    values = frame.to_numpy(dtype=float)
    row_min = values.min(axis=1, keepdims=True)
    row_max = values.max(axis=1, keepdims=True)
    scale = np.where((row_max - row_min) == 0, 1.0, row_max - row_min)
    return (values - row_min) / scale




def draw_annotated_heatmap(ax, raw_frame, norm_frame, title, cmap='YlGnBu'):
    """Draw a heatmap with actual values written on top of color cells."""
    image = ax.imshow(norm_frame, aspect='auto', cmap=cmap, vmin=0, vmax=1)
    ax.set_xticks(np.arange(raw_frame.shape[1]))
    ax.set_xticklabels(raw_frame.columns, rotation=20, ha='right', fontsize=11)
    ax.set_yticks(np.arange(raw_frame.shape[0]))
    ax.set_yticklabels(raw_frame.index, fontsize=11)
    for row_index in range(raw_frame.shape[0]):
        for col_index in range(raw_frame.shape[1]):
            value = raw_frame.iloc[row_index, col_index]
            text_color = 'white' if norm_frame[row_index, col_index] > 0.6 else 'black'
            ax.text(col_index, row_index, f'{value:.2f}', ha='center', va='center', fontsize=10.5, fontweight='bold', color=text_color)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.grid(False)
    return image




fig = plt.figure(figsize=(24, 16), dpi=DISPLAY_DPI, constrained_layout=True, facecolor='white')
gs = fig.add_gridspec(2, 2, hspace=0.30, wspace=0.20)
ax_metrics = fig.add_subplot(gs[0, 0])
ax_crop = fig.add_subplot(gs[0, 1])
ax_parity = fig.add_subplot(gs[1, 0])
ax_range = fig.add_subplot(gs[1, 1])


fig.suptitle(
    f"Ensemble Summary | Avg R²={ensemble_df['Ensemble_R2'].mean():.3f}, "
    f"MAE={ensemble_df['Ensemble_MAE'].mean():.1f}, "
    f"MAPE={ensemble_df['Ensemble_MAPE'].mean():.2f}%, "
    f"sMAPE={ensemble_df['Ensemble_sMAPE'].mean():.2f}%, "
    f"MASE={ensemble_df['Ensemble_MASE'].mean():.2f}",
    fontsize=18,
    fontweight='bold'
 )


# Fold-level metrics heatmap
fold_heatmap = ensemble_df.set_index('Fold')[[
    'Ensemble_R2', 'Ensemble_MAE', 'Ensemble_MAPE', 'Ensemble_sMAPE', 'Ensemble_MASE'
]].T
fold_norm = row_normalize(fold_heatmap)
im1 = draw_annotated_heatmap(ax_metrics, fold_heatmap, fold_norm, 'Fold-Level Metrics')
ax_metrics.set_xlabel('Fold', fontsize=12, fontweight='bold')
ax_metrics.set_ylabel('Metric', fontsize=12, fontweight='bold')
ax_metrics.set_yticklabels(['R²', 'MAE', 'MAPE %', 'sMAPE %', 'MASE'], fontsize=11)
cbar1 = fig.colorbar(im1, ax=ax_metrics, fraction=0.046, pad=0.04, label='Row-normalized color scale')
cbar1.ax.tick_params(labelsize=10)


# Per-crop metrics heatmap
crop_heatmap = crop_perf_df.set_index('Crop')[[
    'Ens_R2', 'Ens_MAE', 'Ens_MAPE', 'Ens_sMAPE', 'Ens_MASE'
]]
crop_norm = row_normalize(crop_heatmap)
im2 = draw_annotated_heatmap(ax_crop, crop_heatmap, crop_norm, 'Per-Crop Metrics')
ax_crop.set_xlabel('Metric', fontsize=12, fontweight='bold')
ax_crop.set_ylabel('Crop', fontsize=12, fontweight='bold')
ax_crop.set_xticklabels(['R²', 'MAE', 'MAPE %', 'sMAPE %', 'MASE'], fontsize=11)
cbar2 = fig.colorbar(im2, ax=ax_crop, fraction=0.046, pad=0.04, label='Row-normalized color scale')
cbar2.ax.tick_params(labelsize=10)


# Actual vs predicted values
all_true = np.concatenate([arr for _, arr in ensemble_all_y_true])
all_pred = np.concatenate([arr for _, arr in ensemble_all_y_pred])
ax_parity.scatter(all_true, all_pred, s=24, alpha=0.45, color='#2c7fb8', edgecolors='none')
min_val = float(min(all_true.min(), all_pred.min()))
max_val = float(max(all_true.max(), all_pred.max()))
ax_parity.plot([min_val, max_val], [min_val, max_val], '--', color='black', linewidth=1.6, label='Perfect prediction')
ax_parity.set_title('Actual vs Predicted Yield', fontsize=14, fontweight='bold')
ax_parity.set_xlabel('Actual Yield (kg/ha)', fontsize=12, fontweight='bold')
ax_parity.set_ylabel('Predicted Yield (kg/ha)', fontsize=12, fontweight='bold')
ax_parity.tick_params(labelsize=10)
ax_parity.grid(alpha=0.3)
ax_parity.legend(fontsize=10)


# Crop range boxplots
id_to_crop = {v: k for k, v in crop_to_id.items()}
crop_names = []
true_boxes = []
pred_boxes = []
for cid in sorted(id_to_crop):
    crop_name = id_to_crop[cid]
    true_vals = []
    pred_vals = []
    for (c_test_arr, y_true_arr), (_, y_pred_arr) in zip(ensemble_all_y_true, ensemble_all_y_pred):
        mask = c_test_arr == cid
        if np.any(mask):
            true_vals.extend(y_true_arr[mask])
            pred_vals.extend(y_pred_arr[mask])
    if true_vals:
        crop_names.append(crop_name)
        true_boxes.append(np.array(true_vals))
        pred_boxes.append(np.array(pred_vals))


positions = np.arange(len(crop_names))
offset = 0.19
bp_true = ax_range.boxplot(true_boxes, positions=positions - offset, widths=0.30, patch_artist=True, showfliers=False)
bp_pred = ax_range.boxplot(pred_boxes, positions=positions + offset, widths=0.30, patch_artist=True, showfliers=False)


for patch in bp_true['boxes']:
    patch.set(facecolor='#2c7fb8', alpha=0.58)
for patch in bp_pred['boxes']:
    patch.set(facecolor='#de2d26', alpha=0.50)
for element in ['whiskers', 'caps', 'medians']:
    plt.setp(bp_true[element], color='#2c7fb8', linewidth=1.4)
    plt.setp(bp_pred[element], color='#de2d26', linewidth=1.4)


ax_range.set_xticks(positions)
ax_range.set_xticklabels(crop_names, rotation=20, ha='right', fontsize=11)
ax_range.set_title('Yield Range by Crop', fontsize=14, fontweight='bold')
ax_range.set_ylabel('Yield (kg/ha)', fontsize=12, fontweight='bold')
ax_range.tick_params(labelsize=10)
ax_range.grid(axis='y', alpha=0.3)
ax_range.legend([bp_true['boxes'][0], bp_pred['boxes'][0]], ['Observed', 'Predicted'], fontsize=10)


# Optional high-res export for reports/screenshots
fig.savefig('../results/ensemble_dashboard_highres.png', dpi=SAVE_DPI, bbox_inches='tight')


plt.show()

In [ ]:
# Export trained fold models for true ensemble evaluation
from pathlib import Path

MODELS_DIR = Path('../models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

fold_artifact_rows = []
for fold_id, fold_model in sorted(fold_models.items()):
    fold_path = MODELS_DIR / f'tcn_mlp_fold_{fold_id}.keras'
    fold_model.save(fold_path)
    fold_artifact_rows.append({
        'Fold': int(fold_id),
        'Path': str(fold_path),
        'Is_Best_Fold': bool(fold_id == best_fold),
    })
    print(f'Saved fold {fold_id} to {fold_path}')

best_fold_path = MODELS_DIR / 'TCN_MLP_ENSEMBLE_best_fold.keras'
if best_fold in fold_models:
    fold_models[best_fold].save(best_fold_path)
    print(f'Saved best fold {best_fold} to {best_fold_path}')

fold_artifacts_df = pd.DataFrame(fold_artifact_rows).sort_values('Fold').reset_index(drop=True)
fold_artifacts_df.to_csv(MODELS_DIR / 'tcn_mlp_fold_artifacts.csv', index=False)
fold_artifacts_df